# Exercise 03.04 (Assignment #01) — 3D Linear Transformations

In [ ]:
import numpy as np

## Points — a square pyramid

4 base corners forming a flat square (z=0), plus 1 apex centered above the base and raised up. Each row is one 3D point (x, y, z).

In [ ]:
P = np.array([
    [0, 0, 0],       # middle
    [1, 0, 0],       # base corner
    [1, 1, 0],       # base corner
    [0, 1, 0],       # base corner
    [0.5, 0.5, 1],   # apex, centered above the base, raised up
])

print("Original pyramid points:")
print(P)

<img src="../Assets/Exercise 03.04 /SquarePyr.png" alt="Original pyramid" width="300">

## Scaling transformation

Diagonal entries = how much to stretch each axis (2x here, uniformly). Off-diagonal zeros mean the axes don't affect each other — x only depends on x, etc.

In [ ]:
T_scale = np.array([[2, 0, 0],
                    [0, 2, 0],
                    [0, 0, 2]])

print("Scaling matrix T_scale:")
print(T_scale)

## Applying the transformation

`T_scale` is 3x3 because it transforms a single 3D point (3 in, 3 out) — not the whole `(5,3)` table at once. The same 3x3 matrix gets reused once per row of `P`.

`T_scale @ point` does 3 dot products, one per row of `T_scale`, against the point — each dot product becomes one coordinate of the new point:

(example) For point 2 with row `(1, 0, 0)`:
- **Row 1** `[2,0,0]` . `(1,0,0)` = `(2×1) + (0×0) + (0×0)` = `2` -> new x
- **Row 2** `[0,2,0]` . `(1,0,0)` = `(0×1) + (2×0) + (0×0)` = `0` -> new y
- **Row 3** `[0,0,2]` . `(1,0,0)` = `(0×1) + (0×0) + (2×0)` = `0` -> new z

Stacking those three results gives the new point: `(2, 0, 0)`.

### Compact version (list comprehension)

In [ ]:
transformed_P = [tuple(coord for coord in T_scale @ point) for point in P]

print("Scaled points:")
print(np.array(transformed_P))

### Expanded version (explicit loops, easier to read)

Same computation as above, spelled out step by step: an outer loop over points, an inner loop over that point's 3 coordinates.

In [ ]:
transformed_P = []              # will collect all 5 transformed points

for point in P:                 # go through P one point (one row) at a time
    new_point = T_scale @ point # apply the scaling matrix to this single point

    clean_point = []            # will hold this point's 3 coordinates, as floats
    for value in new_point:
        clean_point.append(value)

    transformed_P.append(tuple(clean_point))  # append the transformed point

print("Scaled points:")
print(np.array(transformed_P))

<img src="../Assets/Exercise 03.04 /SquarePyr_scaled.png" alt="Scaled pyramid" width="300">

## Rotation transformation

Same 90-degree counter-clockwise rotation matrix from Exercise 03.03, extended to 3D: the bottom row `[0,0,1]` leaves z completely untouched, so this only spins points around in the flat x-y plane.

In [ ]:
T_rotate = np.array([[0, -1, 0],
                      [1,  0, 0],
                      [0,  0, 1]])

print("Rotation matrix T_rotate:")
print(T_rotate)

In [ ]:
transformed_P = []                # will collect all 5 rotated points

for point in P:                   # go through P one point (one row) at a time
    new_point = T_rotate @ point  # apply the rotation matrix to this single point

    clean_point = []              # will hold this point's 3 coordinates, as floats (no truncation)
    for value in new_point:
        clean_point.append(value)

    transformed_P.append(tuple(clean_point))  # save the finished point

print("Rotated points:")
print(np.array(transformed_P))

**Fixed:** the apex `(0.5, 0.5, 1)` now correctly rotates to `(-0.5, 0.5, 1)`, matching the hand calculation exactly, since the `int()` truncation has been removed. All transformed points are now kept as exact floats instead of being silently rounded down.

**Red = original, blue = rotated.**

<img src="../Assets/Exercise 03.04 /SquarePyr_rotated.png" alt="Original (red) vs rotated (blue) pyramid" width="300">

## Shear transformation

Unlike `T_scale` (purely diagonal) or `T_rotate`, row 2 here is `[1,1,0]` instead of `[0,1,0]` — the new y-coordinate depends on both the old y *and* the old x, not just y alone. A point's y-value gets "dragged along" by however far it already is along x, which skews the shape instead of scaling or spinning it.

In [ ]:
T_shear = np.array([[1, 0, 0],
                     [1, 1, 0],
                     [0, 0, 1]])

print("Shear matrix T_shear:")
print(T_shear)

transformed_P = []               # will collect all 5 sheared points

for point in P:                  # go through P one point (one row) at a time
    new_point = T_shear @ point  # apply the shear matrix to this single point

    clean_point = []             # will hold this point's 3 coordinates, as floats (no truncation)
    for value in new_point:
        clean_point.append(value)

    transformed_P.append(tuple(clean_point))  # save the finished point

print("Sheared points:")
print(np.array(transformed_P))

**Red = original, blue = sheared.**

<img src="../Assets/Exercise 03.04 /SquarePyr_Shear.png" alt="Original (red) vs sheared (blue) pyramid" width="300">

## Interpretation

- **Scaling** stretched every point uniformly by 2x — same shape, just bigger, since the diagonal matrix scales each axis independently by the same factor.
- **Rotation** spun the base 90° around the z-axis while leaving height untouched. It's a *rigid* transformation where size and shape are preserved, only orientation changes.
- **Shearing** was the only one that actually distorted the shape: y-values got dragged along by x (`row 2 = [1,1,0]`), so `(0,1,0)` didn't move but `(1,1,0)` shifted all the way to `(1,2,0)`, skewing the angles.
- **Combining:** since each is just a matrix, chaining transformations is matrix multiplication (`T2 @ (T1 @ point) = (T2 @ T1) @ point`). Order matters — just like `A @ T` vs `T @ A` gave different results back in 03.03 — so rotate-then-shear ≠ shear-then-rotate.